# 行/列提取
$A\in R^{m\times n}$，下面$e_i, e_j$为列向量
- 提取矩阵第 i 行：$v = e_i^T A$，$e_i \in R^{m}$
- 提取部分行：比如要提取1，3，5，就构造$mask=[e_1, e_3, e_5]$，result = $mask^T$ @ $A$
- 提取矩阵第 j 列：$u = Ae_j$，$e_j \in R^{n}$
- 提取部分列：比如要提取2，4，6，就构造$mask=[e_2, e_4, e_6]$，result = $A$ @ $mask$

In [3]:
import numpy as np

A = np.random.randint(0, 10, size=(10, 3))
i, j, k = 0, 0, 0
rows_idx = [1, 3 ,5]
# 提取单行
row_i = A[i]         

# 提取多行
rows = A[[i, j, k], :]  # 第i,j,k行
rows = A[rows_idx, :]   # rows_idx为索引数组

# 提取列
col_j = A[:, j]      # 第j列
cols = A[:, [j, k]]  # 第j,k列

# 向量化常用

In [ ]:
# ========================================== 构造one-hot ==================================================== #
# 比如说，现在有一个向量y，形状是(样本数，类别数)，每个位置是该样本所属的类别
# e.g. y = [0, 9, 2, ..., 1]^T, 0 ≤ y[i] < num_classes
# 现在有一个概率矩阵P，形状是(样本数，类别数)，记录了每个样本的分类概率
# 希望提取每个样本所属真实类别的概率，所以要构造一个矩阵，每一列为one-hot，形状为(样本数，类别数)，仅真实类别处值为1

num_items = 3
num_classes = 10
y = np.array([0, 9, 2])
one_hot_mat = np.zeros((num_items, num_classes))
one_hot_mat[np.arange(num_items), y] = 1
one_hot_mat

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
# ========================================== 构造mask PART 1 ================================================ #
# 比如说，现在要给数据加padding，但是我们不希望padding也参与最后的loss计算
vocal_size = 5050
padding_id = vocal_size + 1
example_data = np.array([5052, 12, 33, 105, 5053, 5051, 5051, 5051, 5051])
mask = (example_data != padding_id)
# loss = mask * loss
mask

array([ True,  True,  True,  True,  True, False, False, False, False])

In [ ]:
# ========================================== 构造mask PART 2 ================================================= #
# 比如说，transformer的当前块应该只能看到自己和之前的块，所以我们需要构造上三角的因果mask
def extract_upper_triangle(A):
    mask = np.triu(np.ones_like(A, dtype=bool))
    return A[mask]

# 向量化版本
def extract_upper_triangle_vectorized(A):
    n = A.shape[0]
    U = np.triu(np.ones((n, n)))
    return (A * U)[U.astype(bool)]


# 矩阵形式和求和形式之间的转换

在推导机器学习和数值计算公式的时候，经常会遇到把求和式写成其对应的矩阵形式，或者把矩阵形式拆开写成求和式
然后利用级数或者矩阵的一些性质来继续

内积（两个序列对应相乘求和）：$$\sum_{i=1}^n x_i y_i = x^T y = y^T x$$

平方和/二范数（自身内积）：$$\sum_{i=1}^n x_i^2 = x^T x = \|x\|_2^2$$

二次型（带权重的交叉求和）：$$\sum_{i=1}^n \sum_{j=1}^n A_{ij} x_i x_j = x^T A x$$

矩阵乘法（元素的定义）： 矩阵 $C = AB$ 的第 $i$ 行第 $j$ 列元素是：$$C_{ij} = \sum_{k} A_{ik} B_{kj}$$


代码与数学的“互译”训练

平时在深度学习框架（如 PyTorch 或 NumPy）中写代码时，把你写的 torch.matmul、@ 或 Broadcasting 操作，反向翻译成带有 $\sum$ 的数学公式；反之，看到论文里的复杂求和公式，思考如果不用 for 循环，如何只用矩阵乘法写出高效代码。这种工程实践与数学理论的对齐，是建立直觉的最快方式。

学习爱因斯坦求和约定 (Einstein Summation Convention, Einsum)

这是连接求和式和矩阵式的终极桥梁。在数学中和代码里（torch.einsum）都存在。它强制你思考维度的流转逻辑。例如，矩阵乘法 $C_{ij} = \sum_k A_{ik} B_{kj}$ 在 Einsum 中仅仅写作 ik, kj -> ij。习惯了这种表达，你会对求和符号的去留极为敏感。

## TODO： 推导
岭回归 (Ridge Regression) 的解析解推导。

支持向量机 (SVM) 对偶问题中的二次规划推导。

纯推导一遍多层感知机 (MLP) 的反向传播算法（Jacobian 矩阵是如何把误差一层层用矩阵相乘传回来的）。

# 矩阵求导的布局

矩阵的计算里，形状的转变过程很重要。在求导的时候，先思考求导以后的矩阵应该是什么布局（这也是为什么讲矩阵求导的书里面常说有分子布局和分母布局）

## TODO：记录几个矩阵布局的例子